# 03 - LSTM si CNN-LSTM pe strict benchmark si clean benchmark

Acest notebook antreneaza modele secventiale pe aceleasi scenarii definite in notebookul 02:

- `all_eligible`: test strict pe toate bateriile eligibile;
- `clean_benchmark`: test principal pe baterii cu degradare mai coerenta.

Modelele sunt comparate cu baseline-urile clasice salvate in notebookul 02.

## 1. Imports si paths

In [ ]:
from __future__ import annotations

from copy import deepcopy
from pathlib import Path
import json
import random

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

ARTIFACTS_DIR = REPO_ROOT / "artifacts"
FEATURE_TABLE_PATH = ARTIFACTS_DIR / "features" / "battery_cycle_features_v2.csv"
FEATURE_COLUMNS_PATH = ARTIFACTS_DIR / "features" / "baseline_feature_columns_v2.json"
SCENARIO_CONFIG_PATH = ARTIFACTS_DIR / "splits" / "modeling_scenarios_v1.json"
MODEL_DIR = ARTIFACTS_DIR / "models"
METRICS_DIR = ARTIFACTS_DIR / "metrics"
PRED_DIR = ARTIFACTS_DIR / "predictions"
FIG_DIR = ARTIFACTS_DIR / "figures" / "sequence"
for d in [MODEL_DIR, METRICS_DIR, PRED_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (12, 6)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

assert FEATURE_TABLE_PATH.exists(), "Run notebook 02 first to create the feature table."
assert FEATURE_COLUMNS_PATH.exists(), "Run notebook 02 first to save feature columns."
assert SCENARIO_CONFIG_PATH.exists(), "Run notebook 02 first to create modeling scenarios."

## 2. Incarcare feature table, feature-uri si scenarii

In [ ]:
model_df = pd.read_csv(FEATURE_TABLE_PATH, parse_dates=["start_time", "imp_start_time"])
feature_config = json.loads(FEATURE_COLUMNS_PATH.read_text())
scenarios = json.loads(SCENARIO_CONFIG_PATH.read_text())

TARGET_COL = feature_config["target_col"]
GROUP_COL = feature_config["group_col"]
feature_cols = feature_config["feature_cols"]
SCENARIOS_TO_RUN = ["all_eligible", "clean_benchmark", "nasa_classic_4"]

rows = []
for scenario_name in SCENARIOS_TO_RUN:
    split = scenarios[scenario_name]["split"]
    for split_name, key in [("train", "train_batteries"), ("validation", "validation_batteries"), ("test", "test_batteries")]:
        ids = split[key]
        sub = model_df[model_df[GROUP_COL].isin(ids)]
        rows.append({"scenario": scenario_name, "split": split_name, "batteries": len(ids), "rows": len(sub), "battery_ids": ", ".join(ids)})
display(pd.DataFrame(rows))
print("Feature count:", len(feature_cols))

## 3. Dataset, modele si functii de training

In [ ]:
SEQ_LEN = 20
BATCH_SIZE = 128
EPOCHS = 50
PATIENCE = 8
LR = 1e-3
WEIGHT_DECAY = 1e-4

class SequenceDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class LSTMRegressor(nn.Module):
    def __init__(self, n_features: int, hidden_size: int = 96, num_layers: int = 2, dropout: float = 0.25):
        super().__init__()
        self.lstm = nn.LSTM(input_size=n_features, hidden_size=hidden_size, num_layers=num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
        self.head = nn.Sequential(nn.LayerNorm(hidden_size), nn.Linear(hidden_size, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, 1))

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze(-1)


class CNNLSTMRegressor(nn.Module):
    def __init__(self, n_features: int, conv_channels: int = 64, hidden_size: int = 96, dropout: float = 0.25):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(n_features, conv_channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(conv_channels),
            nn.Dropout(dropout),
            nn.Conv1d(conv_channels, conv_channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(conv_channels),
        )
        self.lstm = nn.LSTM(input_size=conv_channels, hidden_size=hidden_size, num_layers=1, batch_first=True)
        self.head = nn.Sequential(nn.LayerNorm(hidden_size), nn.Linear(hidden_size, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, 1))

    def forward(self, x):
        z = self.conv(x.transpose(1, 2)).transpose(1, 2)
        out, _ = self.lstm(z)
        return self.head(out[:, -1, :]).squeeze(-1)


def regression_metrics(y_true, y_pred) -> dict[str, float]:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {"RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))), "MAE": float(mean_absolute_error(y_true, y_pred)), "R2": float(r2_score(y_true, y_pred))}


def build_sequences(df: pd.DataFrame, scaled_feature_cols: list[str], seq_len: int):
    X_list = []
    y_list = []
    meta_rows = []
    for battery_id, group in df.groupby(GROUP_COL):
        g = group.sort_values("cycle_index").reset_index(drop=True)
        Xg = g[scaled_feature_cols].to_numpy(dtype=np.float32)
        yg = g[TARGET_COL].to_numpy(dtype=np.float32)
        for end_idx in range(len(g)):
            start_idx = max(0, end_idx - seq_len + 1)
            window = Xg[start_idx : end_idx + 1]
            padded = np.zeros((seq_len, Xg.shape[1]), dtype=np.float32)
            padded[-len(window) :] = window
            X_list.append(padded)
            y_list.append(yg[end_idx])
            meta_rows.append(
                {
                    "battery_id": battery_id,
                    "cycle_index": int(g.loc[end_idx, "cycle_index"]),
                    "start_time": g.loc[end_idx, "start_time"],
                    "capacity_ah_clean": float(g.loc[end_idx, "capacity_ah_clean"]),
                    "capacity_ratio": float(g.loc[end_idx, "capacity_ratio"]),
                    TARGET_COL: float(yg[end_idx]),
                }
            )
    return np.stack(X_list), np.asarray(y_list, dtype=np.float32), pd.DataFrame(meta_rows)


def predict_loader(model: nn.Module, loader: DataLoader):
    model.eval()
    preds = []
    truths = []
    losses = []
    criterion = nn.SmoothL1Loss()
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            pred = model(xb)
            loss = criterion(pred, yb)
            preds.append(pred.detach().cpu().numpy())
            truths.append(yb.detach().cpu().numpy())
            losses.append(float(loss.item()) * len(xb))
    pred_log = np.concatenate(preds)
    true_log = np.concatenate(truths)
    pred_raw = np.expm1(np.clip(pred_log, -20, 20))
    true_raw = np.expm1(np.clip(true_log, -20, 20))
    avg_loss = float(np.sum(losses) / len(loader.dataset))
    return avg_loss, pred_raw, true_raw


def train_one_model(model_name: str, model: nn.Module, train_loader: DataLoader, val_loader: DataLoader):
    model = model.to(device)
    criterion = nn.SmoothL1Loss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    best_state = None
    best_val_rmse = np.inf
    best_epoch = 0
    bad_epochs = 0
    history = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad(set_to_none=True)
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        train_loss, train_pred_raw, train_true_raw = predict_loader(model, train_loader)
        val_loss, val_pred_raw, val_true_raw = predict_loader(model, val_loader)
        train_metrics = regression_metrics(train_true_raw, train_pred_raw)
        val_metrics = regression_metrics(val_true_raw, val_pred_raw)
        row = {
            "model": model_name,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "train_RMSE": train_metrics["RMSE"],
            "val_RMSE": val_metrics["RMSE"],
            "train_MAE": train_metrics["MAE"],
            "val_MAE": val_metrics["MAE"],
            "val_R2": val_metrics["R2"],
        }
        history.append(row)

        if val_metrics["RMSE"] < best_val_rmse:
            best_val_rmse = val_metrics["RMSE"]
            best_epoch = epoch
            best_state = deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1

        if epoch == 1 or epoch % 5 == 0:
            print(f"{model_name} epoch {epoch:03d}: train RMSE={train_metrics['RMSE']:.2f}, val RMSE={val_metrics['RMSE']:.2f}")
        if bad_epochs >= PATIENCE:
            print(f"{model_name}: early stopping at epoch {epoch}; best epoch {best_epoch}, val RMSE={best_val_rmse:.2f}")
            break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history), {"best_epoch": best_epoch, "best_val_RMSE": best_val_rmse}

## 4. Rulare modele secventiale pe scenarii

In [ ]:
all_histories = []
validation_summaries = []
test_rows = []
prediction_frames = []

for scenario_name in SCENARIOS_TO_RUN:
    print("\n" + "=" * 80)
    print("Scenario:", scenario_name)
    split = scenarios[scenario_name]["split"]
    scenario_df = model_df[model_df[GROUP_COL].isin(scenarios[scenario_name]["battery_ids"])].copy()
    train_df = scenario_df[scenario_df[GROUP_COL].isin(split["train_batteries"])].copy()
    val_df = scenario_df[scenario_df[GROUP_COL].isin(split["validation_batteries"])].copy()
    test_df = scenario_df[scenario_df[GROUP_COL].isin(split["test_batteries"])].copy()

    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    scaler.fit(imputer.fit_transform(train_df[feature_cols]))
    scaled_feature_cols = [f"z_{c}" for c in feature_cols]

    def add_scaled_features(df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        out[scaled_feature_cols] = scaler.transform(imputer.transform(out[feature_cols]))
        return out

    train_df = add_scaled_features(train_df)
    val_df = add_scaled_features(val_df)
    test_df = add_scaled_features(test_df)

    X_train, y_train_raw, meta_train = build_sequences(train_df, scaled_feature_cols, SEQ_LEN)
    X_val, y_val_raw, meta_val = build_sequences(val_df, scaled_feature_cols, SEQ_LEN)
    X_test, y_test_raw, meta_test = build_sequences(test_df, scaled_feature_cols, SEQ_LEN)
    y_train = np.log1p(y_train_raw)
    y_val = np.log1p(y_val_raw)
    y_test = np.log1p(y_test_raw)

    print("Shapes:", X_train.shape, X_val.shape, X_test.shape)
    preprocessor_path = MODEL_DIR / f"sequence_{scenario_name}_preprocessor_v1.joblib"
    joblib.dump({"imputer": imputer, "scaler": scaler, "feature_cols": feature_cols, "seq_len": SEQ_LEN}, preprocessor_path)

    generator = torch.Generator().manual_seed(SEED)
    train_loader = DataLoader(SequenceDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True, generator=generator)
    val_loader = DataLoader(SequenceDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(SequenceDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

    n_features = X_train.shape[-1]
    model_specs = {
        "LSTM": LSTMRegressor(n_features=n_features),
        "CNN_LSTM": CNNLSTMRegressor(n_features=n_features),
    }

    for model_name, model in model_specs.items():
        full_model_name = f"{scenario_name}__{model_name}"
        trained, history_df, summary = train_one_model(full_model_name, model, train_loader, val_loader)
        history_df["scenario"] = scenario_name
        history_df["base_model"] = model_name
        all_histories.append(history_df)
        validation_summaries.append({"scenario": scenario_name, "model": model_name, **summary})

        _, pred_raw, true_raw = predict_loader(trained, test_loader)
        test_rows.append({"scenario": scenario_name, "model": model_name, "split": "test", **regression_metrics(true_raw, pred_raw)})

        pred_df = meta_test.copy()
        pred_df["scenario"] = scenario_name
        pred_df["model"] = model_name
        pred_df["true_rul"] = true_raw
        pred_df["prediction"] = pred_raw
        pred_df["error"] = pred_df["prediction"] - pred_df["true_rul"]
        pred_df["abs_error"] = pred_df["error"].abs()
        prediction_frames.append(pred_df)

        model_path = MODEL_DIR / f"sequence_{scenario_name}_{model_name.lower()}_state.pt"
        torch.save({"scenario": scenario_name, "model_name": model_name, "state_dict": trained.state_dict(), "feature_cols": feature_cols, "seq_len": SEQ_LEN, "n_features": n_features}, model_path)

    naive_pred = np.full_like(y_test_raw, fill_value=float(y_train_raw.mean()), dtype=float)
    test_rows.append({"scenario": scenario_name, "model": "NaiveTrainMean", "split": "test", **regression_metrics(y_test_raw, naive_pred)})

history_all = pd.concat(all_histories, ignore_index=True)
validation_summary = pd.DataFrame(validation_summaries).sort_values(["scenario", "best_val_RMSE"])
sequence_metrics = pd.DataFrame(test_rows).sort_values(["scenario", "RMSE"])
sequence_predictions = pd.concat(prediction_frames, ignore_index=True)

history_all.to_csv(METRICS_DIR / "sequence_training_history.csv", index=False)
validation_summary.to_csv(METRICS_DIR / "sequence_validation_summary.csv", index=False)
sequence_metrics.to_csv(METRICS_DIR / "sequence_test_metrics.csv", index=False)
sequence_predictions.to_csv(PRED_DIR / "sequence_test_predictions.csv", index=False)

display(validation_summary)
display(sequence_metrics)

## 5. Comparatie cu baseline-urile clasice

In [ ]:
baseline_metrics_path = METRICS_DIR / "baseline_test_metrics.csv"
comparison_parts = [sequence_metrics.assign(family="sequence")]
if baseline_metrics_path.exists():
    baseline_metrics = pd.read_csv(baseline_metrics_path)
    comparison_parts.append(baseline_metrics.assign(family="baseline"))
comparison = pd.concat(comparison_parts, ignore_index=True).sort_values(["scenario", "RMSE"])
comparison.to_csv(METRICS_DIR / "all_model_test_comparison.csv", index=False)
display(comparison)

for scenario_name in SCENARIOS_TO_RUN:
    plot_df = comparison[(comparison["scenario"] == scenario_name) & (~comparison["model"].str.contains("Naive", case=False, na=False))].sort_values("RMSE")
    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ["#7a5c99" if fam == "sequence" else "#3a6ea5" for fam in plot_df["family"]]
    ax.bar(plot_df["model"], plot_df["RMSE"], color=colors)
    ax.set_title(f"{scenario_name}: final test RMSE comparison")
    ax.set_xlabel("Model")
    ax.set_ylabel("RMSE (cycles)")
    ax.tick_params(axis="x", rotation=20)
    plt.tight_layout()
    fig_path = FIG_DIR / f"01_{scenario_name}_all_models_test_rmse_comparison.png"
    plt.savefig(fig_path, dpi=180, bbox_inches="tight")
    print("Saved:", fig_path)
    plt.show()

## 6. Grafice pentru cel mai bun model secvential pe fiecare scenariu

In [ ]:
for scenario_name in SCENARIOS_TO_RUN:
    scenario_seq_metrics = sequence_metrics[(sequence_metrics["scenario"] == scenario_name) & (sequence_metrics["model"] != "NaiveTrainMean")]
    best_sequence_model = scenario_seq_metrics.sort_values("RMSE").iloc[0]["model"]
    best_pred = sequence_predictions[(sequence_predictions["scenario"] == scenario_name) & (sequence_predictions["model"] == best_sequence_model)].copy()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].scatter(best_pred["true_rul"], best_pred["prediction"], s=18, alpha=0.55)
    line_min = float(min(best_pred["true_rul"].min(), best_pred["prediction"].min()))
    line_max = float(max(best_pred["true_rul"].max(), best_pred["prediction"].max()))
    axes[0].plot([line_min, line_max], [line_min, line_max], color="red", linestyle="--", linewidth=1)
    axes[0].set_title(f"{scenario_name}: {best_sequence_model} predicted vs true")
    axes[0].set_xlabel("True RUL (cycles)")
    axes[0].set_ylabel("Predicted RUL (cycles)")
    axes[1].hist(best_pred["error"], bins=35, color="#7a5c99")
    axes[1].axvline(0, color="red", linestyle="--", linewidth=1)
    axes[1].set_title("Prediction error distribution")
    axes[1].set_xlabel("Prediction error (cycles)")
    plt.tight_layout()
    fig_path = FIG_DIR / f"02_{scenario_name}_best_sequence_pred_vs_true.png"
    plt.savefig(fig_path, dpi=180, bbox_inches="tight")
    print("Saved:", fig_path)
    plt.show()

    rows = []
    for battery_id, g in best_pred.groupby("battery_id"):
        rows.append({"scenario": scenario_name, "model": best_sequence_model, "battery_id": battery_id, "rows": len(g), **regression_metrics(g["true_rul"], g["prediction"])})
    bm = pd.DataFrame(rows).sort_values("MAE")
    display(bm)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(bm["battery_id"], bm["MAE"], color="#7a5c99")
    ax.set_title(f"{scenario_name}: {best_sequence_model} test MAE by battery")
    ax.set_xlabel("Battery ID")
    ax.set_ylabel("MAE (cycles)")
    ax.tick_params(axis="x", rotation=30)
    plt.tight_layout()
    fig_path = FIG_DIR / f"03_{scenario_name}_best_sequence_mae_by_battery.png"
    plt.savefig(fig_path, dpi=180, bbox_inches="tight")
    print("Saved:", fig_path)
    plt.show()

    test_battery_ids = sorted(best_pred["battery_id"].unique())
    cols = 2
    rows_n = int(np.ceil(len(test_battery_ids) / cols))
    fig, axes = plt.subplots(rows_n, cols, figsize=(14, 4 * rows_n), squeeze=False)
    for ax, battery_id in zip(axes.ravel(), test_battery_ids):
        g = best_pred[best_pred["battery_id"] == battery_id].sort_values("cycle_index")
        ax.plot(g["cycle_index"], g["true_rul"], label="true RUL", linewidth=2)
        ax.plot(g["cycle_index"], g["prediction"], label="predicted RUL", linewidth=1.6)
        ax.set_title(battery_id)
        ax.set_xlabel("Cycle index")
        ax.set_ylabel("RUL (cycles)")
        ax.legend(fontsize=8)
    for ax in axes.ravel()[len(test_battery_ids):]:
        ax.axis("off")
    plt.tight_layout()
    fig_path = FIG_DIR / f"04_{scenario_name}_best_sequence_rul_curves_by_test_battery.png"
    plt.savefig(fig_path, dpi=180, bbox_inches="tight")
    print("Saved:", fig_path)
    plt.show()

## 7. Interpretare

Daca `clean_benchmark` arata mult mai bine decat `all_eligible`, interpretarea nu este ca am cosmetizat rezultatele, ci ca am separat doua intrebari diferite:

- cat de bine generalizeaza modelul pe toate bateriile eterogene disponibile;
- cat de bine functioneaza pe baterii cu curbe de degradare suficient de coerente pentru un benchmark RUL.

Pentru lucrare, se raporteaza ambele. Pentru demo, se foloseste de obicei `clean_benchmark`, deoarece produce predictii mai stabile si mai usor de explicat.